In [6]:
import sys
print(sys.executable)  # sanity check — confirms venv kernel is active

from dotenv import load_dotenv
load_dotenv(override=True)

/home/vikas/Coding_play_ground/AI/ai-engineering-learing/.venv/bin/python


True

In [7]:
from langchain_groq import ChatGroq
from langchain_mistralai import ChatMistralAI

In [8]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=200
)

In [36]:
prompt = 'Explain me fiber in simple terms'

In [38]:
llm.invoke(prompt).content

'Fiber is a type of nutrient that\'s really good for your body. Here\'s what you need to know:\n\n**What is fiber?**\nFiber is a part of the plants we eat, like fruits, vegetables, whole grains, and legumes (like beans and lentils). It\'s the tough, stringy part that our bodies can\'t break down or digest.\n\n**Why is fiber important?**\nEven though our bodies can\'t digest fiber, it still does a lot of good. Here are some benefits:\n\n1. **Helps with digestion**: Fiber helps move food through your digestive system and prevents constipation (when you have trouble pooping).\n2. **Keeps you full**: Fiber-rich foods tend to be more filling, so you might eat less and feel fuller for longer.\n3. **Lowers cholesterol**: Soluble fiber (found in foods like oats, barley, and fruits) can help lower your "bad" cholesterol levels.\n4. **Helps control blood'

In [42]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain the concept of {topic} in simple words."
)

formatted_prompt = prompt.invoke({"topic": "LangChain"})

print(formatted_prompt.text)

Explain the concept of LangChain in simple words.


In [44]:
# Role Based Prompting
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced SDE. Explain concepts in a beginner-friendly way."),
    ("human", "{question}")
])

messages = prompt.invoke({
    "question": "What is Dependency Injection?"
})

print(messages)

messages=[SystemMessage(content='You are an experienced SDE. Explain concepts in a beginner-friendly way.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is Dependency Injection?', additional_kwargs={}, response_metadata={})]


In [48]:
# Few Shot Prompting
from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate
)

examples = [
    {"input": "Apple", "output": "Fruit"},
    {"input": "Carrot", "output": "Vegetable"},
    {"input": "Rose", "output": "Flower"},
]

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nCategory: {output}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Input: {input}\nCategory:",
    input_variables=["input"]
)

prompt = few_shot_prompt.format(input="Potato")

print(prompt)

Input: Apple
Category: Fruit

Input: Carrot
Category: Vegetable

Input: Rose
Category: Flower

Input: Potato
Category:


In [52]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Translate English to Hindi
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following English text into Hindi:\n\n{text}"
)

# Step 2: Summarize the Hindi text
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following Hindi text in 3-4 lines:\n\n{text}"
)

parser = StrOutputParser()

# Chain : Translation and summarization
chain = translate_prompt | llm | parser | summary_prompt | llm | parser

# Execute the chains sequentially
english_text = """
LangChain is an open-source framework.
"""

summary = chain.invoke({"text": english_text})

print(summary)

लैंगचेन एक ओपन-सोर्स फ्रेमवर्क है जो विकास के लिए खुला है। यह एक तकनीकी ढांचा प्रदान करता है। इसका उपयोग विभिन्न अनुप्रयोगों में किया जा सकता है। यह खुला और लचीला है।


In [55]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

parser = StrOutputParser()

prompt1=ChatPromptTemplate.from_template(
        "Translate the following text into Hindi:\n\n{text}"
    )

# Chain 1: Translate to Hindi
translate_chain = prompt1 | llm | parser

prompt2=ChatPromptTemplate.from_template(
        "Summarize the following text into 5 bullet points:\n\n{text}"
    )

# Chain 2: Generate bullet points
bullet_chain = prompt2 | llm | parser

# Execute both chains in parallel
parallel_chain = RunnableParallel(
    translation=translate_chain,
    bullets=bullet_chain
)

response = parallel_chain.invoke({
    "text": "LangChain is an open-source framework that simplifies building applications powered by Large Language Models."
})

print("Translation:")
print(response["translation"])

print("\nBullet Points:")
print(response["bullets"])

Translation:
लैंगचेन एक ओपन-सोर्स फ्रेमवर्क है जो बड़े भाषा मॉडल द्वारा संचालित अनुप्रयोगों के निर्माण को सरल बनाता है।

Bullet Points:
Here are five bullet points summarizing the text:

* LangChain is an open-source framework.
* The framework is designed to simplify the development process.
* LangChain is focused on building applications.
* These applications are powered by Large Language Models.
* The framework aims to make it easier to build and integrate language models into applications.


In [57]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

parser = StrOutputParser()

prompt1=ChatPromptTemplate.from_template(
        "Reply with only one word: Positive or Negative.\n\nReview: {review}"
    )
prompt2=ChatPromptTemplate.from_template(
        "Write a thank-you message for this customer review:\n\n{review}"
    )
prompt3=ChatPromptTemplate.from_template(
        "Write a polite apology and promise to improve based on this review:\n\n{review}"
    )


# Classify sentiment
sentiment_chain = prompt1 | llm | parser

# Positive response
positive_chain = prompt2 | llm | parser

# Negative response
negative_chain = prompt3 | llm | parser

# Conditional branching
conditional_chain = sentiment_chain | RunnableBranch(
    (
        lambda sentiment: "Positive" in sentiment,
        positive_chain,
    ),
    negative_chain,  # Default branch
)

response = conditional_chain.invoke({
    # "review": "The product quality is amazing and delivery was super fast!"
    "review": "The product quality not good and delivery was loo late!"
})

print(response)

Dear valued customer,

I am deeply sorry to hear that your experience with us was negative. We appreciate you taking the time to share your feedback, and I want to assure you that we take all concerns seriously.

Please know that we are committed to providing the best possible experience for our customers, and it's clear that we fell short in your case. I want to apologize for any inconvenience or disappointment this may have caused.

We are dedicated to learning from our mistakes and using them as an opportunity to grow and improve. We will take your feedback into consideration and work to make necessary changes to ensure that our customers receive the level of service and quality they deserve.

Once again, I apologize for the negative experience, and I hope you will give us the chance to serve you better in the future. If you have any further feedback or concerns, please don't hesitate to reach out to us.

Thank you for your feedback, and we look forward to the opportunity to serve y

In [1]:
from pydantic import BaseModel, Field

class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="One of: positive, negative, neutral")
    key_issues: list[str] = Field(description="Specific problems or praise mentioned")
    summary: str = Field(description="One-sentence summary of the review")

In [11]:
structured_model = llm.with_structured_output(ReviewAnalysis)

result = structured_model.invoke(
    "Review: 'Battery dies in 2 hours, but the screen is gorgeous.'"
)
print(result)
print(type(result))

sentiment='neutral' key_issues=['battery life'] summary='The product has a short battery life but a beautiful screen.'
<class '__main__.ReviewAnalysis'>


In [21]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)

prompt = ChatPromptTemplate.from_template(
    "Analyze this review.\n{review}\n\n{format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
result = chain.invoke({"review": "Battery dies in 2 hours, but the screen is gorgeous."})
print(result)

sentiment='neutral' key_issues=['Battery life is short', 'Screen quality is good'] summary='The product has a short battery life, but it has a gorgeous screen.'


In [23]:
from langchain_core.exceptions import OutputParserException

try:
    result = chain.invoke({"review": "Helllo"})
except OutputParserException as e:
    print("Failed to parse:", e)
    # retry, log, or fall back to a default

Failed to parse: Invalid json output: ## Analysis of the Review

The provided text is not a review in the classical sense but rather an explanation of how a JSON instance should be formatted to conform to a given schema.

### Key Points

1. **Schema Explanation**: The text explains how to create a well-formatted JSON instance that adheres to a specific schema. It provides an example schema and demonstrates a correctly formatted JSON object according to that schema.
2. **Example Schema**: The schema provided includes properties for "sentiment," "key_issues," and "summary," with specific requirements for each, including data types and descriptions.
3. **Required Fields**: The schema specifies that "sentiment," "key_issues," and "summary" are all required fields for any JSON instance claiming to conform to this schema.

### Issues with the Provided Text as a Review

- **Lack of Sentiment Expression**: The text does not express a sentiment (positive, negative, neutral) towards any product,